## Lesson 1 : How Machines read Text-Tokenization
**Before any neural network sees your text,it must be converted to numbers. This Pipeline is called Tokenization, and its very first thing every LLM does with your input.**

# 🧠 The Core Problem
__Neural Networks only understands numbers so we need answers:__

How do you convert string " Hello I Love LLms " into numbers a model can process?

**There are 3 Historical approaches, each solving previous one problems.**

# Approach 1: Character Level Tokenization
# __Splits text into individual characters__

In [5]:
text=" Hello I Love LLms"
token=list(text)
vocab=sorted(set(token))
char_to_id={ch: i for i, ch in enumerate(vocab) }
print(token)

print(char_to_id)

ids = [char_to_id[c] for c in token]
print(ids)

[' ', 'H', 'e', 'l', 'l', 'o', ' ', 'I', ' ', 'L', 'o', 'v', 'e', ' ', 'L', 'L', 'm', 's']
{' ': 0, 'H': 1, 'I': 2, 'L': 3, 'e': 4, 'l': 5, 'm': 6, 'o': 7, 's': 8, 'v': 9}
[0, 1, 4, 5, 5, 7, 0, 2, 0, 3, 7, 9, 4, 0, 3, 3, 6, 8]


__Problem : The word "Love" is 4 tokens. A 1000 word essay becomes 5000+ tokens. Sequences get very long, and model never learns that a "L"+"o"+"v"+"e"= a meaningful unit.__

-----------------------------------------------------------------------------------------------




# Approach 2: Word-level Tokenization
__Split on spaces and punctuation.__


In [10]:
import re
text=" Hello, I Love LLms!,they loved llms"
token=re.findall(r"\b\w+\b",text.lower())
vocab=sorted(set(token))
word_to_id={w:i for i,w in enumerate(vocab)}
print(token)
print(word_to_id)
ids=[word_to_id[w] for w in token]
print(ids)

['hello', 'i', 'love', 'llms', 'they', 'loved', 'llms']
{'hello': 0, 'i': 1, 'llms': 2, 'love': 3, 'loved': 4, 'they': 5}
[0, 1, 3, 2, 5, 4, 2]


__Probelm : "love","loved","loving","lovely",they are all different tokens- model never learns they share a root. rare words like "Chatgpt" or "Anthropic" becomes unknown. Real World vocubalaries need 100k+ entries.__

-----------------------------------------------------------------------------------------------------




# Approach 3: Subword-Tokenization(What LLMs actually use)
__The insight : split/rare unknown words into pieces, keep words whole.__

- "Unhappiness" -> ["un","happiness"]
- "Chatgpt" -> ["chat", "g", "pt"]
- "The" -> ["The"](common, kept whole)

# This gives you:
- a small vocubalary (~32k - 100k tokens) that covers all possible text.
- Meaningful sub-units(Prefixes,suffixes,root)
- No unknown words - Everything is splittable into known pieces.

__The main algorithm used is called Byte-Pair-Encoding(BPE)__


In [18]:
# How BPE works
from collections import Counter, defaultdict
import re
def get_vocab(text):
    vocab=Counter()
    for word in text.lower().split():#re.findall(r"\b\w+\b", corpus.lower())
        vocab[' '.join(list(word))+' </w>']+=1
    return vocab
def get_pairs(vocab):
    pairs=defaultdict(int)
    for word,freq in vocab.items():
        symbols=word.split()
        for i in range(len(symbols)-1):
            pairs[(symbols[i],symbols[i+1])]+=freq
    return pairs

def merge_vocab(pair,vocab):
    new_vocab={}
    bigram=' '.join(pair)
    replacement=''.join(pair)
    for word in vocab:
        new_word=word.replace(bigram,replacement)
        new_vocab[new_word]=vocab[word]
    return new_vocab


text="low lower newest widest low low lower"
vocab=get_vocab(text)
print("Initial vocab:")
for word,freq in vocab.items():
    print(f"{word}: {freq}")

print("\n-- Running 8 BPE Merge steps --\n")
for step in range(8):
    pairs=get_pairs(vocab)
    if not pairs:
        break
    best_pair=max(pairs,key=pairs.get)
    print(f"step {step+1}: merging pair{best_pair} (freq:{pairs[best_pair]})")
    vocab=merge_vocab(best_pair,vocab)
print("\nFinal vocab:")
for word in vocab:
    print(word)
    




Initial vocab:
l o w </w>: 3
l o w e r </w>: 2
n e w e s t </w>: 1
w i d e s t </w>: 1

-- Running 8 BPE Merge steps --

step 1: merging pair('l', 'o') (freq:5)
step 2: merging pair('lo', 'w') (freq:5)
step 3: merging pair('low', '</w>') (freq:3)
step 4: merging pair('low', 'e') (freq:2)
step 5: merging pair('lowe', 'r') (freq:2)
step 6: merging pair('lower', '</w>') (freq:2)
step 7: merging pair('e', 's') (freq:2)
step 8: merging pair('es', 't') (freq:2)

Final vocab:
low</w>
lower</w>
n e w est </w>
w i d est </w>


# Now we use real thing Hugging Face Tokenizers


In [3]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained("gpt2")
text=" Hello, I Love LLms!,they loved llms"
tokens=tokenizer.tokenize(text)
ids=tokenizer.encode(text)
back=tokenizer.decode(ids)

print("Tokens:",tokens)
print("IDs:",ids)
print("Decoded:",back)
print("Vocab size:",tokenizer.vocab_size)

Tokens: ['ĠHello', ',', 'ĠI', 'ĠLove', 'ĠLL', 'ms', '!,', 'they', 'Ġloved', 'Ġll', 'ms']
IDs: [18435, 11, 314, 5896, 27140, 907, 28265, 9930, 6151, 32660, 907]
Decoded:  Hello, I Love LLms!,they loved llms
Vocab size: 50257


In [4]:
# Some edge cases
print("\n-- some edge cases --\n")
for word in ["ChatGPT", "tokenization", "😊", "LLaVA", "unhappiness"]:
    t=tokenizer.tokenize(word)
    print(f"{word:20s} → {t}")


-- some edge cases --

ChatGPT              → ['Chat', 'G', 'PT']
tokenization         → ['token', 'ization']
😊                    → ['ðŁĺ', 'Ĭ']
LLaVA                → ['LL', 'a', 'VA']
unhappiness          → ['un', 'h', 'appiness']


# The BERT Tokenizer (it uses WordPiece,a BPE variant)

In [10]:
bert_tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")
text="I love building LLMs from scratch!"
tokens=bert_tokenizer.tokenize(text)
ids=bert_tokenizer.encode(text)
print("\nBERT Tokens:",tokens)
print("BERT IDs:",ids)


BERT Tokens: ['i', 'love', 'building', 'll', '##ms', 'from', 'scratch', '!']
BERT IDs: [101, 1045, 2293, 2311, 2222, 5244, 2013, 11969, 999, 102]


# Notice the difference
- GPT-2 uses G^ to mark spaces (its BPE on Bytes)
- BERT Lower cases everything (it's uncased)
- Bert adds special tokens [CLS]=101 , [SEP]=102


# 🧠 Important Questions about Tokenization
__Q1. Why does GPT4 Charge by tokens not by words ?__
 - Because Tokens are the models actual unit of computation. A token ~ 0.75 words on average in english. Code and non-english text uses more tokens per word.

 __Q2. Why does "1+1=2" sometimes confuse LLMs ?__
 - Numbers get split unpredictably. "1000000" migh be ["100', "000", "0"]-> three separate tokens model must arithmetically combine. This is a fundamental limitation.

 __Q3. Whats the difference between BPE,Wordpiece and sentancepiece ?__
 - BPE(GPT family) : Merges most frequent pairs greedily
 - Wordpiece(BERT) : Merges pairs that maximize liklihood of training data
 - SentencePiece(T5,LLaMA) : treats spaces as regular characters, works on raw unicode--language-agnostic, no pre tokenization needed

--------------------------------------------------------------------------------------------------

# Answer Yourself
- Why can't we just use word-level tokenization for LLMs?
- What problem does BPE solve, and what's the core algorithm idea?
- Run the Hugging Face code above — how many tokens does "unhappiness" become in GPT-2 vs BERT?
- What are special tokens like [CLS], [SEP], [PAD] and why do they exist?

-----------------------------------------------------------------------------------------------------

